# 🎨 Data Designer 101: Credit Card Transaction Datasets

In this notebook, we will demonstrate the basics of `DataDesigner` by generating four related datasets for a credit card transaction system: Cardholder, Card, Merchant, and Bank datasets.

<br>

### 💾 Install `gretel-client` and its dependencies

In [1]:
%%capture
%pip install -U gretel_client

## ⚙️ Initialize Data Designer with a Model Suite

- `DataDesigner` uses "Model Suites" to group LLMs based on the permissiveness of their licenses.

- Example Model Suites include "apache-2.0" (fully permissive) and "llama-3.x" (llama community license agreement).

In [2]:
from gretel_client.navigator_client import Gretel

# We import AIDD column and parameter types using this shorthand for convenience.
import gretel_client.data_designer.params as P
import gretel_client.data_designer.columns as C

# The Gretel object is the SDK's main entry point for interacting with Gretel's API.
gretel = Gretel(api_key="prompt")

Found cached Gretel credentials
Logged in as kornfield@gretel.ai ✅
Using project: kornfield-e164e
Project link: https://console-eng.gretel.ai/proj_2ts0pRfscRdl8BHqmeI958jQuRk


## 🎲 Creating the Cardholder Dataset

Let's start by creating the Cardholder dataset with demographics, credit information, and risk flags.

In [4]:
# Create a new Data Designer instance for Cardholder dataset
cardholder_aidd = gretel.data_designer.new(model_suite="apache-2.0")

# Add cardholder_id as a unique identifier
cardholder_aidd.add_column(
    C.SamplerColumn(
        name="cardholder_id",
        type=P.SamplerType.UUID,
        params=P.UUIDSamplerParams()
    )
)

# Add demographics
cardholder_aidd.add_column(
    C.SamplerColumn(
        name="person",
        type=P.SamplerType.PERSON,
        params=P.PersonSamplerParams(age_range=[18, 85])
    )
)

# Add country
cardholder_aidd.add_column(
    C.SamplerColumn(
        name="country",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["United States", "Canada", "United Kingdom", "Germany", "France", "Australia", "Japan", "Singapore"]
        )
    )
)

# Add credit limit
cardholder_aidd.add_column(
    C.SamplerColumn(
        name="credit_limit",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=1000, high=50000),
        convert_to="int"
    )
)

# Add credit score
cardholder_aidd.add_column(
    C.SamplerColumn(
        name="credit_score",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=300, high=850),
        convert_to="int"
    )
)

# Add historical spend statistics
cardholder_aidd.add_column(
    C.SamplerColumn(
        name="avg_1day_spend",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=10, high=500),
        convert_to="float"
    )
)

cardholder_aidd.add_column(
    C.SamplerColumn(
        name="avg_7day_spend",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=50, high=2000),
        convert_to="float"
    )
)

cardholder_aidd.add_column(
    C.SamplerColumn(
        name="avg_30day_spend",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=200, high=8000),
        convert_to="float"
    )
)

cardholder_aidd.add_column(
    C.SamplerColumn(
        name="txn_count_30days",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=5, high=100),
        convert_to="int"
    )
)

# Add risk flags
cardholder_aidd.add_column(
    C.SamplerColumn(
        name="prior_fraud_flag",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["Yes", "No"],
            weights=[0.05, 0.95]  # 5% have prior fraud
        )
    )
)

cardholder_aidd.add_column(
    C.SamplerColumn(
        name="high_risk_location",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["Yes", "No"],
            weights=[0.1, 0.9]  # 10% in high risk locations
        )
    )
)

cardholder_aidd.validate()

[13:45:25] [INFO] Validation passed ✅


DataDesigner(
    model_suite: apache-2.0
    sampler_columns: [
        "cardholder_id",
        "person",
        "country",
        "credit_limit",
        "credit_score",
        "avg_1day_spend",
        "avg_7day_spend",
        "avg_30day_spend",
        "txn_count_30days",
        "prior_fraud_flag",
        "high_risk_location"
    ]
)

## 🏪 Creating the Merchant Dataset

Now let's create the Merchant dataset with location and category information.

In [8]:
# Create a new Data Designer instance for Merchant dataset
merchant_aidd = gretel.data_designer.new(model_suite="apache-2.0")

# Add merchant_id as a unique identifier
merchant_aidd.add_column(
    C.SamplerColumn(
        name="merchant_id",
        type=P.SamplerType.UUID,
        params=P.UUIDSamplerParams()
    )
)

# Add merchant name using LLM
merchant_aidd.add_column(
    C.LLMTextColumn(
        name="merchant_name",
        prompt=(
            "Generate a realistic merchant name for a business. It should be a company name that could exist in the real world. "
            "The merchant will be located in {{ country }} , and in the category/subcategory of {{ merchant_category }}/{{ merchant_category }}"
            "Respond with only the merchant name, no other text."
        ),
        system_prompt=(
            "You are a helpful assistant that generates realistic merchant names. "
            "You respond with only the merchant name, no other text."
        )
    )
)

# Add location coordinates
merchant_aidd.add_column(
    C.SamplerColumn(
        name="latitude",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=25.0, high=49.0),  # Continental US
        convert_to="float"
    )
)

merchant_aidd.add_column(
    C.SamplerColumn(
        name="longitude",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=-125.0, high=-66.0),  # Continental US
        convert_to="float"
    )
)

# Add country
merchant_aidd.add_column(
    C.SamplerColumn(
        name="country",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["United States", "Canada", "United Kingdom", "Germany", "France", "Australia", "Japan", "Singapore"]
        )
    )
)

# Add merchant category
merchant_aidd.add_column(
    C.SamplerColumn(
        name="merchant_category",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["retail", "food_beverage", "travel", "entertainment", "healthcare", "automotive", "professional_services"]
        )
    )
)

# Add subcategory based on main category
merchant_aidd.add_column(
    C.SamplerColumn(
        name="merchant_subcategory",
        type=P.SamplerType.SUBCATEGORY,
        params=P.SubcategorySamplerParams(
            category="merchant_category",
            values={
                "retail": ["electronics", "clothing", "home_garden", "sports", "jewelry"],
                "food_beverage": ["restaurants", "fast_food", "grocery", "coffee_shops", "bars"],
                "travel": ["hotels", "airlines", "car_rental", "travel_agencies", "cruises"],
                "entertainment": ["movies", "gaming", "sports_events", "concerts", "museums"],
                "healthcare": ["hospitals", "pharmacies", "dental", "vision", "specialists"],
                "automotive": ["dealerships", "repair_shops", "gas_stations", "parts_stores", "car_wash"],
                "professional_services": ["legal", "accounting", "consulting", "real_estate", "insurance"]
            }
        )
    )
)

# Add global risk score
merchant_aidd.add_column(
    C.SamplerColumn(
        name="global_risk_score",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=1, high=100),
        convert_to="int"
    )
)

merchant_aidd.validate()

[13:52:10] [INFO] Validation passed ✅


DataDesigner(
    model_suite: apache-2.0
    sampler_columns: [
        "merchant_id",
        "latitude",
        "longitude",
        "country",
        "merchant_category",
        "merchant_subcategory",
        "global_risk_score"
    ]
    llm_text_columns: ['merchant_name']
)

## 🏦 Creating the Bank Dataset

Finally, let's create the Bank dataset with regional and risk information.

In [17]:
# Create a new Data Designer instance for Bank dataset
bank_aidd = gretel.data_designer.new(model_suite="apache-2.0")

# Add bank_id as a unique identifier
bank_aidd.add_column(
    C.SamplerColumn(
        name="bank_id",
        type=P.SamplerType.UUID,
        params=P.UUIDSamplerParams()
    )
)

# Add bank name using LLM
bank_aidd.add_column(
    C.LLMTextColumn(
        name="bank_name",
        prompt=(
            "Generate a realistic bank name. It should be a financial institution name that could exist in the real world. "
            "It will be located in the region {{ region }} and the country {{ country }}"
            "Respond with only the bank name, no other text."
        ),
        system_prompt=(
            "You are a helpful assistant that generates realistic bank names. "
            "You respond with only the bank name, no other text."
        )
    )
)

# Add country
bank_aidd.add_column(
    C.SamplerColumn(
        name="country", # TODO: Country and region have to be generated in a dependent manner
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["United States", "Canada", "United Kingdom", "Germany", "France", "Australia", "Japan", "Singapore"]
        )
    )
)

# Add region
bank_aidd.add_column(
    C.SamplerColumn(
        name="region",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["North America", "Europe", "Asia Pacific", "Latin America", "Middle East", "Africa"]
        )
    )
)

# Add risk score
bank_aidd.add_column(
    C.SamplerColumn(
        name="risk_score",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=1, high=100),
        convert_to="int"
    )
)

# Add market share percentage
bank_aidd.add_column(
    C.SamplerColumn(
        name="market_share_percent",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=0.1, high=25.0),
        convert_to="float"
    )
)

# Add compliance flags
bank_aidd.add_column(
    C.SamplerColumn(
        name="aml_compliance_flag",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["compliant", "under_review", "non_compliant"],
            weights=[0.85, 0.1, 0.05]
        )
    )
)

bank_aidd.add_column(
    C.SamplerColumn(
        name="kyc_compliance_flag",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["compliant", "under_review", "non_compliant"],
            weights=[0.9, 0.08, 0.02]
        )
    )
)

bank_aidd.add_column(
    C.SamplerColumn(
        name="regulatory_oversight_flag",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["none", "low", "medium", "high"],
            weights=[0.3, 0.4, 0.2, 0.1]
        )
    )
)

bank_aidd.validate()

[13:57:36] [INFO] Validation passed ✅


DataDesigner(
    model_suite: apache-2.0
    sampler_columns: [
        "bank_id",
        "country",
        "region",
        "risk_score",
        "market_share_percent",
        "aml_compliance_flag",
        "kyc_compliance_flag",
        "regulatory_oversight_flag"
    ]
    llm_text_columns: ['bank_name']
)

In [18]:
# Preview Cardholder dataset
print("=== CARDHOLDER DATASET PREVIEW ===")
cardholder_preview = cardholder_aidd.preview()
print(cardholder_preview.dataset.df.head())
print("\n")

# Preview Bank dataset
print("=== BANK DATASET PREVIEW ===")
bank_preview = bank_aidd.preview()
print(bank_preview.dataset.df.head())


# Preview Merchant dataset
print("=== MERCHANT DATASET PREVIEW ===")
merchant_preview = merchant_aidd.preview()
print(merchant_preview.dataset.df.head())
print("\n")

=== CARDHOLDER DATASET PREVIEW ===
[13:57:43] [INFO] 🚀 Generating preview
[13:57:44] [INFO] 🎲 Step 1: Using samplers to generate 11 columns
[13:57:45] [INFO] 🎉 Your dataset preview is ready!
                      cardholder_id  \
0  9e16659bc31f42a698ffd88186a26280   
1  b372cb756a9d4858b1ea68a368fd0044   
2  3dd62b00e9a040bbbe2a4f2f46190e77   
3  3a37ef1c523241e18fa579a54f4abbc9   
4  b66c7e08f9e740e5a9c61b7acf879715   

                                              person        country  \
0  {'first_name': 'Karen', 'middle_name': 'Ann', ...        Germany   
1  {'first_name': 'Aliki', 'middle_name': 'Alejan...  United States   
2  {'first_name': 'Aj', 'middle_name': '', 'last_...      Australia   
3  {'first_name': 'Richard', 'middle_name': 'T', ...      Australia   
4  {'first_name': 'Jackie', 'middle_name': 'Lynn'...  United States   

   credit_limit  credit_score  avg_1day_spend  avg_7day_spend  \
0         35572           315      222.311508     1619.887988   
1         44087  

In [33]:
# Use previews to create seeds for dependent columns
bank_seed_data = bank_preview.dataset.df["bank_id"].to_frame()
cardholder_seed_data = cardholder_preview.dataset.df["cardholder_id"].to_frame()

# Option 2: Cross join (all combinations)
bank_seed_data_temp = bank_seed_data.copy()
cardholder_seed_data_temp = cardholder_seed_data.copy()
bank_seed_data_temp['key'] = 1
cardholder_seed_data_temp['key'] = 1
combined_cross_join = bank_seed_data_temp.merge(cardholder_seed_data_temp, on='key').drop('key', axis=1)
print(combined_cross_join.head(10))
print(f"Total combinations: {len(combined_cross_join)}")

                            bank_id                     cardholder_id
0  34b36ee937834fa486193058f7d7ef8a  9e16659bc31f42a698ffd88186a26280
1  34b36ee937834fa486193058f7d7ef8a  b372cb756a9d4858b1ea68a368fd0044
2  34b36ee937834fa486193058f7d7ef8a  3dd62b00e9a040bbbe2a4f2f46190e77
3  34b36ee937834fa486193058f7d7ef8a  3a37ef1c523241e18fa579a54f4abbc9
4  34b36ee937834fa486193058f7d7ef8a  b66c7e08f9e740e5a9c61b7acf879715
5  34b36ee937834fa486193058f7d7ef8a  ee882ecba9a64414b8c22a2ce67e9970
6  34b36ee937834fa486193058f7d7ef8a  6c393739ceaa41f7b551d80dd9654f05
7  34b36ee937834fa486193058f7d7ef8a  f4ffd7368da5421392c6b903d992a9fe
8  34b36ee937834fa486193058f7d7ef8a  142d390e28944a0b8a6b167da06bc6ce
9  34b36ee937834fa486193058f7d7ef8a  c5a7d08f0f314dde97f08c5df0aa7811
Total combinations: 100


## 💳 Creating the Card Dataset

Now let's create the Card dataset with card details and status information.

In [34]:
# Create a new Data Designer instance for Card dataset
card_aidd = gretel.data_designer.new(model_suite="apache-2.0")
card_aidd.with_seed_dataset(
    combined_cross_join,
    sampling_strategy="shuffle", 
    with_replacement=True)

# Add card_id as a unique identifier
card_aidd.add_column(
    C.SamplerColumn(
        name="card_id",
        type=P.SamplerType.UUID,
        params=P.UUIDSamplerParams()
    )
)

# Add cardholder_id reference
card_aidd.add_column(
    name="cardholder_id_fk",
    type="expression",
    expr="{{ cardholder_id }} ",
)

# Add issuer bank ID
card_aidd.add_column(
    name="bank_id_fk",
    type="expression",
    expr="{{ bank_id }} ",
)

# Add card type
card_aidd.add_column(
    C.SamplerColumn(
        name="card_type",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["credit", "debit", "prepaid"],
            weights=[0.6, 0.35, 0.05]  # Most are credit cards
        )
    )
)

# Add expiry date (simplified as months from now)
card_aidd.add_column(
    C.SamplerColumn(
        name="expiry_months_from_now",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=1, high=60),
        convert_to="int"
    )
)

# Add CVV check result
card_aidd.add_column(
    C.SamplerColumn(
        name="cvv_check_result",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["pass", "fail", "not_provided"],
            weights=[0.85, 0.1, 0.05]
        )
    )
)

# Add activation date (simplified as days ago)
card_aidd.add_column(
    C.SamplerColumn(
        name="activation_days_ago",
        type=P.SamplerType.UNIFORM,
        params=P.UniformSamplerParams(low=1, high=3650),  # Up to 10 years
        convert_to="int"
    )
)

# Add card status
card_aidd.add_column(
    C.SamplerColumn(
        name="card_status",
        type=P.SamplerType.CATEGORY,
        params=P.CategorySamplerParams(
            values=["active", "blocked", "suspended"],
            weights=[0.92, 0.06, 0.02]
        )
    )
)

card_aidd.validate()

[15:04:38] [INFO] 🌱 Using seed dataset with file ID: file_36b4877c99a340f39ea9785a4f3b051d
[15:04:42] [INFO] Validation passed ✅
[15:04:42] [INFO] Validation passed ✅


DataDesigner(
    model_suite: apache-2.0
    seed_dataset: file_36b4877c99a340f39ea9785a4f3b051d
    seed_columns: ['bank_id', 'cardholder_id']
    sampler_columns: [
        "card_id",
        "card_type",
        "expiry_months_from_now",
        "cvv_check_result",
        "activation_days_ago",
        "card_status"
    ]
    expression_columns: ['cardholder_id_fk', 'bank_id_fk']
)

## 👀 Preview the Datasets

Let's preview each dataset to see how they look.

In [36]:
# Preview Card dataset
print("=== CARD DATASET PREVIEW ===")
card_preview = card_aidd.preview()
print(card_preview.dataset.df.head())
print("\n")

=== CARD DATASET PREVIEW ===
[15:06:16] [INFO] 🚀 Generating preview
[15:06:16] [INFO] 🚀 Generating preview
[15:06:18] [INFO] 🌱 Step 1: Seeding workflow with dataset
[15:06:18] [INFO] 🌱 Step 1: Seeding workflow with dataset
[15:06:20] [INFO] 🎲 Step 2: Using samplers to generate 6 columns
[15:06:20] [INFO] 🎲 Step 2: Using samplers to generate 6 columns
[15:06:20] [INFO] 🔗 Step 3: Concatenating seed and sampler datasets
[15:06:20] [INFO] 🔗 Step 3: Concatenating seed and sampler datasets
[15:06:21] [INFO] 💬 Step 4: Rendering expression column `cardholder_id_fk`
[15:06:21] [INFO] 💬 Step 4: Rendering expression column `cardholder_id_fk`
[15:06:21] [INFO] 💬 Step 5: Rendering expression column `bank_id_fk`
[15:06:21] [INFO] 💬 Step 5: Rendering expression column `bank_id_fk`
[15:06:22] [INFO] 🎉 Your dataset preview is ready!
                            bank_id                     cardholder_id  \
0  b14662c1d9af44c88603ba29bfa4a33b  6c393739ceaa41f7b551d80dd9654f05   
1  141a692c994d4fcfa4aea3a

## 🧐 Adding Evaluation Reports

Let's add evaluation reports to each dataset for quality assessment.

In [43]:
# Add evaluation reports to all datasets
cardholder_aidd.with_evaluation_report()
card_aidd.with_evaluation_report()
merchant_aidd.with_evaluation_report()
bank_aidd.with_evaluation_report()

DataDesigner(
    model_suite: apache-2.0
    sampler_columns: [
        "bank_id",
        "country",
        "region",
        "risk_score",
        "market_share_percent",
        "aml_compliance_flag",
        "kyc_compliance_flag",
        "regulatory_oversight_flag"
    ]
    llm_text_columns: ['bank_name']
)

## 🆙 Scale up to Full Datasets!

Now let's create the full datasets with larger record counts.

In [44]:
# Create Cardholder dataset
print("Creating Cardholder dataset...")
cardholder_workflow = cardholder_aidd.create(num_records=100, name="credit-card-cardholder-dataset")
print(f"Cardholder dataset created: {cardholder_workflow}")

# Create Bank dataset
print("\nCreating Bank dataset...")
bank_workflow = bank_aidd.create(num_records=50, name="credit-card-bank-dataset")
print(f"Bank dataset created: {bank_workflow}")

# Create Merchant dataset
print("\nCreating Merchant dataset...")
merchant_workflow = merchant_aidd.create(num_records=100, name="credit-card-merchant-dataset")
print(f"Merchant dataset created: {merchant_workflow}")


Creating Cardholder dataset...
[15:24:37] [INFO] 🚀 Submitting batch workflow
▶️ Using Workflow: w_3178Z7afeAvjqv8vUTyKBgmK6NK
▶️ Created Workflow Run: wr_317ALUMvQn5pInVSyAb92oK90QP
🔗 Workflow Run console link: https://console-dev.gretel.ai/workflows/w_3178Z7afeAvjqv8vUTyKBgmK6NK/runs/wr_317ALUMvQn5pInVSyAb92oK90QP
Cardholder dataset created: <gretel_client.workflows.workflow.WorkflowRun object at 0x113e13a10>

Creating Bank dataset...
[15:24:41] [INFO] 🚀 Submitting batch workflow
▶️ Using Workflow: w_3178ZRaOBQG1g0z8bNcHAyXWKmb
▶️ Created Workflow Run: wr_317ALzGSdHRo8Pn5ONOyqS4tV8y
🔗 Workflow Run console link: https://console-dev.gretel.ai/workflows/w_3178ZRaOBQG1g0z8bNcHAyXWKmb/runs/wr_317ALzGSdHRo8Pn5ONOyqS4tV8y
Bank dataset created: <gretel_client.workflows.workflow.WorkflowRun object at 0x112f05dd0>

Creating Merchant dataset...
[15:24:46] [INFO] 🚀 Submitting batch workflow
▶️ Using Workflow: w_3178ZuLExJAOOQBtazJXrC1Bqdz
▶️ Created Workflow Run: wr_317AN6WJ93mREa8QFrfQ6zQenNH
🔗 

In [38]:
for workflow in [cardholder_workflow, bank_workflow, merchant_workflow]:
    workflow.wait_until_done()

Fetching task logs for workflow run wr_3178Z2Tfc3CCzH31VHzLaIS90dB
Got task wt_3178Z2g6B68Z1mWR6ehbJ8h82gt
Workflow run is now in status: RUN_STATUS_ACTIVE
[using-samplers-to-generate-11-columns] Task Status is now: RUN_STATUS_ACTIVE
[using-samplers-to-generate-11-columns] 2025-08-10 22:13:44.849096+00:00 Preparing step 'using-samplers-to-generate-11-columns'
[using-samplers-to-generate-11-columns] 2025-08-10 22:13:59.632230+00:00 Starting 'generate_columns_using_samplers' task execution
[using-samplers-to-generate-11-columns] 2025-08-10 22:13:59.634396+00:00 🎲 🧑‍🚀 Initializing person generation
[using-samplers-to-generate-11-columns] 2025-08-10 22:14:20.022423+00:00 🎲 Using numerical samplers to generate 100 records across 11 columns
[using-samplers-to-generate-11-columns] 2025-08-10 22:14:24.873391+00:00 Task 'generate_columns_using_samplers' executed successfully
[using-samplers-to-generate-11-columns] 2025-08-10 22:14:24.873771+00:00 Task execution completed. Saving task outputs.
[

In [40]:
# Use workflows to create seeds for dependent columns
bank_seed_data = bank_workflow.dataset.df["bank_id"].to_frame()
cardholder_seed_data = cardholder_workflow.dataset.df["cardholder_id"].to_frame()

# Cross join (all combinations)
bank_seed_data_temp = bank_seed_data.copy()
cardholder_seed_data_temp = cardholder_seed_data.copy()
bank_seed_data_temp['key'] = 1
cardholder_seed_data_temp['key'] = 1
combined_cross_join = bank_seed_data_temp.merge(cardholder_seed_data_temp, on='key').drop('key', axis=1)
print(combined_cross_join.head(10))
print(f"Total combinations: {len(combined_cross_join)}")
card_aidd.with_seed_dataset(combined_cross_join, sampling_strategy="shuffle", with_replacement=True)

                            bank_id                     cardholder_id
0  70dba76b22ad4a2590d5a1cbce46fdc8  9bc1ec8890ab43ac9ed0a9b8760d14c6
1  70dba76b22ad4a2590d5a1cbce46fdc8  9f64a71f61ea4e439e3b488360c72b80
2  70dba76b22ad4a2590d5a1cbce46fdc8  63f201c9543345d2b707d400524691d6
3  70dba76b22ad4a2590d5a1cbce46fdc8  e3ddfe76466a445da5739cd1adee9bb4
4  70dba76b22ad4a2590d5a1cbce46fdc8  2b83885602474d83ac50eb1db79f66c2
5  70dba76b22ad4a2590d5a1cbce46fdc8  f623a51cb6de4b5c84f9d049695bc72c
6  70dba76b22ad4a2590d5a1cbce46fdc8  1497c7e31c8744b0b297be31530cc4a9
7  70dba76b22ad4a2590d5a1cbce46fdc8  01d919c4a4134ff7b3d7857a7ac98530
8  70dba76b22ad4a2590d5a1cbce46fdc8  54f1065a4c854d8aaf1f8b09a5d8c2c2
9  70dba76b22ad4a2590d5a1cbce46fdc8  a09210f45b9f41cd9e383fe65242c487
Total combinations: 5000
[15:23:33] [INFO] 🌱 Using seed dataset with file ID: file_75760da0301142539e49d74a1ae71c4c


DataDesigner(
    model_suite: apache-2.0
    seed_dataset: file_75760da0301142539e49d74a1ae71c4c
    seed_columns: ['bank_id', 'cardholder_id']
    sampler_columns: [
        "card_id",
        "card_type",
        "expiry_months_from_now",
        "cvv_check_result",
        "activation_days_ago",
        "card_status"
    ]
    expression_columns: ['cardholder_id_fk', 'bank_id_fk']
)

In [48]:
# Create Card dataset
print("\nCreating Card dataset...")
card_workflow = card_aidd.create(num_records=150, name="credit-card-card-dataset")
print(f"Card dataset created: {card_workflow}")


Creating Card dataset...
[15:42:12] [INFO] 🚀 Submitting batch workflow
▶️ Using Workflow: w_317AGF4WPiXgUZRrYOc81q7lIDT
▶️ Created Workflow Run: wr_317CU4cswNCunEEwFvDbops7JDq
🔗 Workflow Run console link: https://console-dev.gretel.ai/workflows/w_317AGF4WPiXgUZRrYOc81q7lIDT/runs/wr_317CU4cswNCunEEwFvDbops7JDq
Card dataset created: <gretel_client.workflows.workflow.WorkflowRun object at 0x113dbc210>


In [45]:
card_workflow.wait_until_done()

Fetching task logs for workflow run wr_317AGLpAKP8in5P4h1sXVPAZiMn
Got task wt_317AGQfxK0we3J71L2zfWOus0gA
Got task wt_317AGOZxO0a4NJ1t9nMpxO4YjQD
Workflow run is now in status: RUN_STATUS_ACTIVE
[seeding-workflow-with-dataset] 2025-08-10 22:24:19.005845+00:00 Preparing step 'seeding-workflow-with-dataset'
[seeding-workflow-with-dataset] 2025-08-10 22:24:29.809008+00:00 Starting 'sample_from_dataset' task execution
[seeding-workflow-with-dataset] 2025-08-10 22:24:29.809574+00:00 🎲 Sampling 150 records from input dataset *with replacement*
[seeding-workflow-with-dataset] 2025-08-10 22:24:29.820708+00:00 Task 'sample_from_dataset' executed successfully
[seeding-workflow-with-dataset] 2025-08-10 22:24:29.821174+00:00 Task execution completed. Saving task outputs.
[seeding-workflow-with-dataset] 2025-08-10 22:24:30.362945+00:00 Task outputs saved.
[using-samplers-to-generate-6-columns] Task Status is now: RUN_STATUS_ACTIVE
[using-samplers-to-generate-6-columns] 2025-08-10 22:29:33.683172+0

In [56]:
import os

for workflow in [cardholder_workflow, bank_workflow, merchant_workflow, card_workflow]:
    file_name = workflow.name + "-report.html"
    workflow.report.download(file_name, format="html")
    current_dir = os.path.join(os.curdir, file_name)
    print(f"Saved report to {file_name}")
# Use another tool to view the reports, i.e., the browser, since VSCode can't display them that well

Saved to credit-card-cardholder-dataset-report.html
Saved to credit-card-bank-dataset-report.html
Saved to credit-card-merchant-dataset-report.html
Saved to credit-card-card-dataset-report.html
